In [ ]:
from option_analyzer import *
from indicators import compute_emas
self = OptionAnalyzer('quotes', 'chain')
pd.set_option("display.max_columns", None)

etf_symbols = 'QQQ|IWM|SPY|GLD|DIA'
mag_symbols = 'NVDA|GOOGL|MSFT|AAPL|META|AMZN|TSLA|TSM'
filter_etf = lambda _df: _df[_df.symbol.str.contains(etf_symbols)]
filter_mag = lambda _df: _df[_df.symbol.str.contains(mag_symbols)]
filter_others = lambda _df: _df[~_df.symbol.str.contains(etf_symbols + '|' + mag_symbols)]

In [ ]:
def show_fastest_decay_puts(dfp, dth_profit_lb=12, dth_strike_margin_lb=0, n_rows=10, W=1000, H=600):
    _dfp = dfp[(dfp.dthStrikeMargin >= dth_strike_margin_lb) & (dfp.dthProfit >= dth_profit_lb)].sort_values(by='dthr').head(n_rows).copy()
    _dfp['option'] = _dfp.symbol + ' ' + _dfp.strike.astype(str).str.replace(r'\.0$', '', regex=True) + ' ' + _dfp.expDt
    px.scatter(_dfp, x='dthr', y='dthProfit', color='option', width=W, height=H).show()
    top_cols = ['option', 'dte', 'dthr', 'dthStrikeMargin', 'dthProfit', 'mid', 'moneyness', 'Delta', 'OpenInterest', 'lastPrice', 'pctSpread']
    return _dfp.loc[:, top_cols + [c for c in _dfp.columns if c not in ['symbol', 'strike', 'expDt'] + top_cols]]

def profit_overview_of_short_puts(dfp, mn_lb=0.9, mn_ub=1.0, dte_ub=40, dth_profit_lb=24):
    mn2df = {}
    metrics = ['dthProfit', 'dte', 'dthStrikeMargin', 'Delta']
    for mn in np.arange(mn_lb, mn_ub, 0.01):
        key = f'{mn}'
        _df = dfp[(dfp.moneyness <= mn) & (dfp.dte <= dte_ub) & (dfp.dthProfit >= dth_profit_lb) & (dfp.dthStrikeMargin >= 0)].reset_index()
        mn2df[key] = _df.loc[_df.groupby('symbol')['dthProfit'].idxmax(), ['symbol'] + metrics].set_index('symbol')
    for metric in metrics:
        _df = pd.DataFrame(dict([(key, mn2df[key][metric]) for key in mn2df])).T
        _df.index.name = 'moneyness'
        title = f'DTE under {dte_ub} days' if metric == 'dte' else f'dthProfit >= {dth_profit_lb} percent' if metric == 'dthProfit' else metric
        #px.bar(_df, barmode='group', title=title, width=1500, height=420).show()
        px.line(_df, title=title, width=1500, height=420).show()

def show_put_options_by_dte_and_moneyness(dfp, dte, mn_lb=0.9, mn_ub=1.0, n_rows=20):
    _filter = (dfp.dte==dte) & (dfp.moneyness >= mn_lb) & (dfp.moneyness <= mn_ub) & (dfp.dthStrikeMargin >= 0)
    _dfp = dfp[_filter].copy()
    if _dfp.shape[0] == 0:
        print('Are you using a valid dpe?')
        return
    title = f'{dte} dte {_dfp.iloc[0].expDt}'
    _dfp = _dfp.sort_values(by='dthProfit', ascending=False).head(n_rows).sort_values(by='dthStrikeMargin', ascending=False)
    px.scatter(_dfp, x='dthStrikeMargin', y='dthProfit', color='symbol', title=title, width=1000, height=600).show()
    top_cols = ['symbol', 'expDt', 'strike', 'dte', 'dthStrikeMargin', 'dthProfit', 'mid', 'moneyness', 'Delta', 'OpenInterest', 'lastPrice', 'pctSpread']
    return _dfp.loc[:, top_cols + [c for c in _dfp.columns if c not in top_cols]]

### Run this once every day to load close prices in the past 60 days

In [ ]:
df_close = pd.read_csv('output/yf_close.csv', parse_dates=['Date']).set_index('Date').tail(60)
df_close.columns.name = 'symbol'
latest_bollinger_file = max(glob('output/bollinger*.csv'))
df_boll = pd.read_csv(latest_bollinger_file).set_index('symbol')
df_boll = df_boll.loc[:, ['LB20', 'UB20', 'MA20', 'LB30', 'UB30', 'MA30']]
today = pd.Timestamp.now().normalize()
print('df_close data age:', today - df_close.index[-1])
print('latest Bollinger file:', latest_bollinger_file)

### Run this cell to read from data directory

In [ ]:
os.system('sync > /dev/null 2>&1')
option_type = 'put'
servers = sorted(set([f.split('~')[1] for f in glob(os.path.expanduser(f'~/lab/data/{option_type}~*~*.csv'))]))
latest_option_files = [sorted(glob(os.path.expanduser(f'~/lab/data/{option_type}~{svr}~*.csv')))[-1] for svr in servers]
print('\n'.join(['%40s' % _ for _ in map(os.path.basename, latest_option_files)]))
chain_file_mtimes = dict([(os.path.basename(_f), os.path.getmtime(_f)) for _f in glob(os.path.expanduser('~/lab/chain/*'))])
latest_symbol = sorted(chain_file_mtimes, key=chain_file_mtimes.get)[-1]
print('Last symbol:', latest_symbol, datetime.fromtimestamp(chain_file_mtimes[latest_symbol]).strftime('%F %T'))

# Read option data processed by servers
dfp = pd.concat([pd.read_csv(_f) for _f in latest_option_files])

# Compute EMA
df_today = dfp.loc[:, ['symbol', 'lastPrice']].drop_duplicates().rename(columns={'lastPrice': today}).set_index('symbol')
if df_today.columns[0] in df_close.T.columns:
    df_price = df_close
else:
    df_price = df_close.T.join(df_today, how='right').T
df_ema = compute_emas(df_price, [21, 50])

# Quick overview of the three major ETFs
dfp_etf = filter_etf(dfp)
dfp_etf =dfp_etf[dfp_etf.dthStrikeMargin >= 1].sort_values(by='dthProfit', ascending=False)
px.scatter(dfp_etf.head(100), x='dthStrikeMargin', y='dthProfit', color='expDt', height=400, width=1500).show()
dfp_etf.head(10)

In [ ]:
__df = dfp[(dfp.symbol=='GLD') & (dfp.strike == 360) & (dfp.dthProfit >= 10)].sort_values(by='dthr')
px.scatter(__df, x='dte', y='dthProfit', width=800).show()
__df.head(20)

In [ ]:
dfp_etf2 = show_fastest_decay_puts(filter_etf(dfp), dth_profit_lb=20, dth_strike_margin_lb=0.5, n_rows=10, W=1000, H=600)
dfp_etf2

In [ ]:
dfp_all = show_fastest_decay_puts(dfp[~dfp.symbol.str.contains(r'MRVL|SNDK|MU')], dth_profit_lb=60, dth_strike_margin_lb=5, n_rows=40, W=1000, H=600)
dfp_all

In [ ]:
show_fastest_decay_puts(filter_mag(dfp), dth_profit_lb=60, dth_strike_margin_lb=0, n_rows=20, W=1000, H=600)

In [ ]:
dthStrikeMargin_LB_mag = 5
dfp_mag = filter_mag(dfp)
#dfp_mag = dfp_mag[dfp_mag.dthr <= 0.25]
dfp_mag = dfp_mag[dfp_mag.dthStrikeMargin >= dthStrikeMargin_LB_mag].sort_values(by='dthProfit', ascending=False)
px.scatter(dfp_mag.head(100), x='dthStrikeMargin', y='dthProfit', color='expDt', height=400, width=1500).show()
dfp_mag.head(10)

In [ ]:
dthStrikeMargin_LB_others = 10
dfp_o = filter_others(dfp)
#dfp_o = dfp_o[dfp_o.dthr <= 0.25]
dfp_o = dfp_o[dfp_o.dthStrikeMargin >= dthStrikeMargin_LB_others].sort_values(by='dthProfit', ascending=False)
px.scatter(dfp_o.head(100), x='dthStrikeMargin', y='dthProfit', color='expDt', height=400, width=1500).show()
print(list(dfp_o.head(10).symbol.unique()))
dfp_o.head(20)

In [ ]:
_dfp = dfp[(dfp.symbol=='SNDK') & (dfp.dte==14) & (dfp.dthr <= 0.5) * (dfp.mid >= 1)].sort_values(by='dthProfit', ascending=False)
px.scatter(_dfp, x='dthr', y='mid', color='strike', width=1000).show()
_dfp

### Overview of put options for three groups of symbols: ETFs, Mag 7 + TMC, and Others

In [ ]:
profit_overview_of_short_puts(filter_others(dfp), 0.72, 0.8, dte_ub=45, dth_profit_lb=50)
profit_overview_of_short_puts(filter_mag(dfp), dte_ub=45, dth_profit_lb=24)
profit_overview_of_short_puts(filter_etf(dfp), dte_ub=45, dth_profit_lb=24)

### ETF dthProfit vs dthStrikeMargin on specific DTE

In [ ]:
target_dte = 2
_dfp_etf = show_put_options_by_dte_and_moneyness(filter_etf(dfp), target_dte, mn_lb=0.9, mn_ub=0.96, n_rows=10)
_dfp_etf

In [ ]:
self.rank_put_spreads(_dfp_etf, risk_limit=100_000, max_oi_ratio=0.1, leg_2_ratio_ub=0.05, buying_power=500_000).head(20)

### Mag 7 + TSM dthProfit vs dthStrikeMargin on specific DTE

In [ ]:
target_dte = 2
_dfp_mag = filter_mag(dfp)
_max_dth_profit = _dfp_mag[_dfp_mag.dthStrikeMargin >= 5].dthProfit.max()
print(_max_dth_profit)
_dfp_mag = _dfp_mag[_dfp_mag.dthProfit >= 0.5*_max_dth_profit]
_dfp_mag = show_put_options_by_dte_and_moneyness(_dfp_mag, target_dte, mn_lb=0.8, mn_ub=0.93, n_rows=20)
_dfp_mag

In [ ]:
self.rank_put_spreads(_dfp_mag, risk_limit=100_000, max_oi_ratio=0.1, leg_2_ratio_ub=0.05, buying_power=500_000).head(20)

### All Other dthProfit vs dthStrikeMargin on specific DTE

In [ ]:
target_dte = 2
_dfp = show_put_options_by_dte_and_moneyness(filter_others(dfp), target_dte, mn_lb=0.8, mn_ub=0.82, n_rows=20)
_dfp

In [ ]:
self.rank_put_spreads(_dfp, risk_limit=100_000, max_oi_ratio=0.1, buying_power=500_000).head(60)

In [ ]:
_symbol = 'TSLA'
px.line(pd.concat([df_ema[_symbol], df_price[_symbol]], axis=1), width=1500, height=700)